# 📦 Dataset Preparation for Kaggle & YOLOv11

This notebook converts the raw Singapore Smart City traffic images scraped on Google Drive into a Kaggle-ready dataset for YOLOv11 training.

**Pipeline:**
1. Mounts Google Drive and reads collected raw images.
2. Performs Auto-labeling using YOLOv11-pretrained to generate initial bounding boxes.
3. Implements stratified train/val/test split (70/15/15) based on camera locations and time-of-day.
4. Generates standard YOLO annotations and `data.yaml`.
5. Zips the final Kaggle-ready `.zip` structure back to Google Drive.

In [ ]:
# 1. Setup Environment
!pip install -q ultralytics pandas scikit-learn pillow

import os
import gc
import json
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from getpass import getpass

# Mount Drive to access raw data
from google.colab import drive
drive.mount('/content/drive')

RAW_DATA_PATH = Path('/content/drive/MyDrive/sg_smart_city/data/raw')
OUTPUT_DATASET = Path('/content/dataset/sg_traffic')
os.makedirs(OUTPUT_DATASET, exist_ok=True)

In [ ]:
# 2. Gather Metadata 
print("Scanning raw data directories...")
metadata_records = []

for jsonl_file in RAW_DATA_PATH.rglob('*.jsonl'):
    with open(jsonl_file, 'r') as f:
        for line in f:
            if not line.strip(): continue
            record = json.loads(line)
            
            # Check if corresponding image actually exists and is readable
            img_rel_path = Path(record['image_path']).name
            img_full_path = jsonl_file.parent / img_rel_path
            
            if img_full_path.exists() and img_full_path.stat().st_size > 0:
                record['absolute_path'] = str(img_full_path)
                # Extract hour for stratification
                dt = pd.to_datetime(record['timestamp'])
                record['hour'] = dt.hour
                metadata_records.append(record)

df = pd.DataFrame(metadata_records)
print(f"\n✅ Found {len(df):,} valid traffic images.")
if len(df) == 0:
    raise ValueError("No data found in Google Drive! Ensure Data Collection has run.")

In [ ]:
# 3. Strict Temporal Split (Zero Data Leakage Protocol)
# ⚠️ Senior Engineer Note: Randomly splitting video frames causes severe data leakage 
# because continuous frames (e.g., 8:01 AM and 8:02 AM) look identical. 
# To prevent the model from artificially memorizing frames, we MUST split strictly by time.

# Convert timestamp to datetime and sort it
df['timestamp_dt'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by='timestamp_dt').reset_index(drop=True)

# 70% Train, 15% Val, 15% Test chronologically
n_total = len(df)
train_idx = int(n_total * 0.70)
val_idx = int(n_total * 0.85)

train_df = df.iloc[:train_idx].copy()
val_df = df.iloc[train_idx:val_idx].copy()
test_df = df.iloc[val_idx:].copy()

print(f"Temporal Boundaries:")
print(f"Train: {train_df['timestamp_dt'].min()} to {train_df['timestamp_dt'].max()} ({len(train_df)} imgs)")
print(f"Val:   {val_df['timestamp_dt'].min()} to {val_df['timestamp_dt'].max()} ({len(val_df)} imgs)")
print(f"Test:  {test_df['timestamp_dt'].min()} to {test_df['timestamp_dt'].max()} ({len(test_df)} imgs)")


In [ ]:
# 4. Prepare YOLO Directories
for split in ['train', 'val', 'test']:
    os.makedirs(OUTPUT_DATASET / 'images' / split, exist_ok=True)
    os.makedirs(OUTPUT_DATASET / 'labels' / split, exist_ok=True)

# Base COCO Classes (mapping to our generic 6-class traffic structure if needed)
# We use standard YOLO pretrained to auto-label vehicles
from ultralytics import YOLO
model = YOLO('yolov11x.pt')  # Use loudest model for best pseudo-labels

def process_split(split_df, split_name):
    print(f"\nProcessing {split_name} split...")
    # Restrict to vehicle classes: 2=car, 3=motorcycle, 5=bus, 7=truck in COCO
    vehicle_classes = [2, 3, 5, 7]
    
    for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
        img_path = row['absolute_path']
        img_name = Path(img_path).name
        dest_img = OUTPUT_DATASET / 'images' / split_name / img_name
        shutil.copy(img_path, dest_img)
        
        # Predict to get YOLO pseudo-labels
        results = model.predict(img_path, verbose=False, classes=vehicle_classes, conf=0.3)[0]
        
        label_path = OUTPUT_DATASET / 'labels' / split_name / f"{Path(img_name).stem}.txt"
        with open(label_path, 'w') as f:
            for box in results.boxes:
                cls = int(box.cls[0].item())
                # Remap to 0-indexed contiguous (0: car, 1: moto, 2: bus, 3: truck)
                mapped_cls = {2: 0, 3: 1, 5: 2, 7: 3}.get(cls, 0)
                xywh = box.xywhn[0].tolist()  # Normalized xywh
                f.write(f"{mapped_cls} {xywh[0]} {xywh[1]} {xywh[2]} {xywh[3]}\n")

process_split(train_df, 'train')
process_split(val_df, 'val')
process_split(test_df, 'test')

In [ ]:
# 5. Create data.yaml
yaml_content = f"""path: {OUTPUT_DATASET}
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""
with open(OUTPUT_DATASET / 'data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml generated!")

In [ ]:
# 6. Package and export to Drive
zip_path = '/content/sg_traffic_dataset.zip'
!cd /content/dataset && zip -r -q {zip_path} sg_traffic

final_dest = '/content/drive/MyDrive/sg_smart_city/data/sg_traffic_dataset.zip'
shutil.copy(zip_path, final_dest)
print(f"\n🚀 Dataset successfully prepared and zipped to:\n{final_dest}")
print("\nYou can now upload this directly to Kaggle Datasets!")